# Une boucle d’entraînement complète

Un seul programme relie données, apprentissage, validation et rechargement.

Nous exécutons ce notebook avec un noyau Python neuf, sur CPU.

Version vérifiée : Python 3.11.9, PyTorch 2.10.0 et TorchAudio 2.10.0.

Les cellules n’installent aucun paquet et ne téléchargent aucune donnée.

Les nombres du cours viennent de cette version ; une autre version peut modifier certains arrondis ou messages.

Dans Colab, choisissons un environnement sans accélérateur, puis Exécution → Tout exécuter.

Les trous des exercices restent dans du Markdown : ils ne cassent donc pas l’exécution complète.

[Relire le chapitre](https://radiocodeur.github.io/quiz/cours.html?theme=pytorch#boucle-entrainement-complete).

Le programme suivant est copié à l’identique depuis le cours.

In [1]:
import os
import sys
import tempfile

# À lancer dans un noyau neuf : ce tutoriel reste sur CPU.
os.environ["CUDA_VISIBLE_DEVICES"] = ""
import torch
torch.set_num_threads(1)
torch.manual_seed(17)
print("Python :", sys.version.split()[0])
print("PyTorch :", torch.__version__)
print("CUDA visible :", torch.cuda.is_available())
_quiz_initial_directory = os.getcwd()
_quiz_temporary_directory = tempfile.TemporaryDirectory(prefix="quiz-pytorch-")
os.chdir(_quiz_temporary_directory.name)
assert not torch.cuda.is_available(), 'Choisir un noyau CPU neuf pour ce notebook.'

Python : 3.11.9
PyTorch : 2.10.0+cpu
CUDA visible : False


## Nous lisons le programme

**1. Construire sans mélanger**

fabriquer crée deux ensembles séparés. Nuages permet leur accès ; assembler conserve les associations entre une ligne et sa cible.

**2. Apprendre, puis observer**

Chaque lot d’entraînement réinitialise les gradients, calcule la perte, dérive et met à jour. La validation utilise eval et no_grad.

**3. Relire les sorties réelles**

Les moyennes sont pondérées par le nombre d’exemples. La sauvegarde contient les poids et la convention de classes.

In [2]:
from pathlib import Path
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader

torch.manual_seed(17)
torch.set_num_threads(1)
device = torch.device("cpu")
perte_neutre = nn.functional.cross_entropy(torch.zeros(1, 2), torch.tensor([0]))
print(f"Perte de deux scores égaux : {perte_neutre.item():.6f}")

def fabriquer(n):
    cibles = torch.arange(n) % 2
    centres = (2 * cibles.float() - 1).unsqueeze(1)
    mesures = 0.4 * torch.randn(n, 3) + centres
    return mesures, cibles

class Nuages(Dataset):
    def __init__(self, mesures, cibles):
        self.mesures, self.cibles = mesures, cibles
    def __len__(self):
        return len(self.cibles)
    def __getitem__(self, indice):
        return self.mesures[indice], self.cibles[indice]

def assembler(exemples):
    mesures, cibles = zip(*exemples)
    return torch.stack(mesures), torch.stack(cibles).long()

train = Nuages(*fabriquer(128))
validation = Nuages(*fabriquer(64))
train_loader = DataLoader(train, batch_size=32, shuffle=True,
                          collate_fn=assembler, num_workers=0)
val_loader = DataLoader(validation, batch_size=32, shuffle=False,
                        collate_fn=assembler, num_workers=0)

def creer_modele():
    return nn.Sequential(nn.Linear(3, 8), nn.ReLU(), nn.Linear(8, 2))

modele = creer_modele().to(device)
critere = nn.CrossEntropyLoss()
optimiseur = torch.optim.Adam(modele.parameters(), lr=0.03)
historique = []
N = 5
for epoch in range(1, N + 1):
    modele.train()
    somme, effectif = 0.0, 0
    for mesures, cibles in train_loader:
        mesures, cibles = mesures.to(device), cibles.to(device)
        optimiseur.zero_grad(set_to_none=True)
        perte = critere(modele(mesures), cibles)
        perte.backward()
        optimiseur.step()
        somme += perte.item() * len(cibles)
        effectif += len(cibles)
    train_loss = somme / effectif

    modele.eval()
    somme, effectif = 0.0, 0
    with torch.no_grad():
        for mesures, cibles in val_loader:
            mesures, cibles = mesures.to(device), cibles.to(device)
            perte = critere(modele(mesures), cibles)
            somme += perte.item() * len(cibles)
            effectif += len(cibles)
    val_loss = somme / effectif
    historique.append([train_loss, val_loss])
    print(f"epoch {epoch}: train={train_loss:.6f}, val={val_loss:.6f}")

chemin = Path("modele-nuages.pt")
torch.save({"model": modele.state_dict(), "classes": ["nuage 0", "nuage 1"]}, chemin)
recharge = creer_modele().to(device)
etat = torch.load(chemin, map_location=device, weights_only=True)
recharge.load_state_dict(etat["model"])
recharge.eval()
with torch.no_grad():
    sonde = validation.mesures[:2].to(device)
    identique = torch.equal(modele(sonde), recharge(sonde))
print("Rechargement identique :", identique)

Perte de deux scores égaux : 0.693147


epoch 1: train=0.783676, val=0.444391
epoch 2: train=0.325945, val=0.156877
epoch 3: train=0.114880, val=0.050986
epoch 4: train=0.040490, val=0.017714
epoch 5: train=0.014973, val=0.007170
Rechargement identique : True


## Interpréter avant de modifier

Exécution CPU vérifiée ; les sorties sont conservées dans scripts/verify/pytorch-additions-results.json.

Le programme contient toutes les fonctions nécessaires, y compris collate_fn.

La validation observe des exemples réservés sans ajuster les paramètres.

Les sorties publiées viennent de l’exécution conservée, pas d’une performance audio supposée.

## Exercice 1 — Assembler un lot et ses cibles

Complétons une fonction collate_fn. Elle reçoit une liste de couples, puis construit deux tenseurs.

Copions ce squelette dans une nouvelle cellule, puis remplaçons les `...`.

```python
def assembler_exemples(exemples):
    mesures, cibles = zip(*exemples)
    return torch....(mesures), torch....(cibles).long()
```

**Piste :** stack ajoute l’axe des exemples. Les indices de classe restent entiers.

## Exercice 2 — Moyenner une perte sur tous les exemples

Deux lots ont des tailles différentes. Complétons une moyenne qui donne le même poids à chaque exemple.

Copions ce squelette dans une nouvelle cellule, puis remplaçons les `...`.

```python
pertes_moyennes = [0.2, 0.8]
tailles = [4, 1]
somme = sum(perte * ... for perte, n in zip(pertes_moyennes, tailles))
moyenne = somme / sum(...)
print(moyenne)
```

**Piste :** Nous retrouvons d’abord la somme de chaque lot, puis divisons par l’effectif total.

In [3]:
os.chdir(_quiz_initial_directory)
_quiz_temporary_directory.cleanup()
print('Programme terminé ; fichiers temporaires supprimés.')

Programme terminé ; fichiers temporaires supprimés.


## Solutions — ouvrons-les après notre recherche

Les solutions utilisent leurs propres variables. Nous pouvons les copier dans de nouvelles cellules.

<details>
<summary>Afficher la solution 1 — Assembler un lot et ses cibles</summary>

<pre><code>import torch
def assembler_exemples_exo(exemples):
    mesures, cibles = zip(*exemples)
    return torch.stack(mesures), torch.stack(cibles).long()
lot_exo = [(torch.tensor([1., 2., 3.]), torch.tensor(0)),
           (torch.tensor([4., 5., 6.]), torch.tensor(1))]
mesures_exo, cibles_exo = assembler_exemples_exo(lot_exo)
assert mesures_exo.shape == (2, 3)
assert cibles_exo.shape == (2,) and cibles_exo.dtype == torch.long
print(&#x27;formes du lot :&#x27;, tuple(mesures_exo.shape), tuple(cibles_exo.shape))</code></pre>

</details>

<details>
<summary>Afficher la solution 2 — Moyenner une perte sur tous les exemples</summary>

<pre><code>pertes_moyennes_exo = [0.2, 0.8]
tailles_exo = [4, 1]
somme_exo = sum(perte * n for perte, n in zip(pertes_moyennes_exo, tailles_exo))
moyenne_exo = somme_exo / sum(tailles_exo)
assert abs(moyenne_exo - 0.32) &lt; 1e-12
print(&#x27;perte moyenne par exemple :&#x27;, moyenne_exo)</code></pre>

</details>

